# Irodori-TTS VoiceDesign Lab

Irodori-TTS の VoiceDesign checkpoint を使って、同じ自己紹介文を複数の話し方で生成するノートブックです。

最初のモデルロードと Hugging Face からのダウンロードは時間がかかります。2 回目以降はキャッシュされます。

In [ ]:
from pathlib import Path
import sys
from IPython.display import Audio, display
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUT_DIR = ROOT / "outputs" / "voice_design_lab"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("root:", ROOT)
print("python:", sys.executable)
print("outputs:", OUT_DIR)
print("torch:", torch.__version__)
print("mps:", torch.backends.mps.is_available())


In [ ]:
try:
    from huggingface_hub import hf_hub_download
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "このノートブックは Irodori TTS Lab kernel で実行してください。"
        "VS Code/Cursor の右上の kernel 選択から `Irodori TTS Lab` を選ぶか、"
        "ターミナルで `uv run python -m ipykernel install --user --name irodori-tts-lab --display-name 'Irodori TTS Lab'` を実行してください。"
    ) from exc

from irodori_tts.inference_runtime import (
    InferenceRuntime,
    RuntimeKey,
    SamplingRequest,
    default_runtime_device,
    save_wav,
)

HF_CHECKPOINT = "Aratako/Irodori-TTS-500M-v2-VoiceDesign"
CODEC_REPO = "Aratako/Semantic-DACVAE-Japanese-32dim"
DEVICE = default_runtime_device()  # Apple Silicon なら通常 mps

checkpoint_path = hf_hub_download(repo_id=HF_CHECKPOINT, filename="model.safetensors")
print("checkpoint:", checkpoint_path)
print("device:", DEVICE)

runtime = InferenceRuntime.from_key(
    RuntimeKey(
        checkpoint=checkpoint_path,
        model_device=DEVICE,
        codec_repo=CODEC_REPO,
        model_precision="fp32",
        codec_device=DEVICE,
        codec_precision="fp32",
    )
)

print("use_caption_condition:", runtime.model_cfg.use_caption_condition)
print("use_speaker_condition:", runtime.model_cfg.use_speaker_condition)


In [ ]:
TEXT = """
はじめまして。私は彩人です。研究と開発を通じて、人の思考を助ける道具を作っています。今日は、音声合成の表現力を確かめるために、同じ自己紹介をいくつかの話し方で読み上げています。
""".strip()

STYLE_PRESETS = {
    "confident_clear": "明るく自信に満ちた若い男性の声。背筋を伸ばして、ハキハキと、語尾まで明瞭に、少し速めのテンポで堂々と自己紹介してください。",
    "confident_calm": "落ち着いた自信のある男性の声。低すぎない自然な声で、聞き手に安心感を与えるように、ゆっくり丁寧に、しかし迷いなく話してください。",
    "nervous_mumbling": "自信がなさそうな若い男性の声。小さめの声で、少し息が漏れるように、ところどころ迷いながら、ボソボソと控えめに自己紹介してください。",
    "shy_soft": "内気でやわらかい男性の声。距離感は近く、声量は控えめで、少し照れながら、優しく自然に話してください。",
}

TEXT


In [ ]:
def synthesize_style(
    name: str,
    caption: str,
    *,
    seed: int = 42,
    num_steps: int = 24,
    cfg_scale_text: float = 3.0,
    cfg_scale_caption: float = 3.5,
) -> Path:
    result = runtime.synthesize(
        SamplingRequest(
            text=TEXT,
            caption=caption,
            no_ref=True,
            seconds=30.0,
            num_steps=num_steps,
            cfg_scale_text=cfg_scale_text,
            cfg_scale_caption=cfg_scale_caption,
            cfg_scale_speaker=1.0,
            cfg_guidance_mode="independent",
            seed=seed,
            trim_tail=True,
        ),
        log_fn=print,
    )
    out_path = OUT_DIR / f"{name}_seed{result.used_seed}.wav"
    save_wav(out_path, result.audio, result.sample_rate)
    print("saved:", out_path)
    print("sample_rate:", result.sample_rate)
    return out_path


In [ ]:
# まずは狙いの 2 種類だけ生成します。
paths = []
for name in ["confident_clear", "nervous_mumbling"]:
    print("\n===", name, "===")
    paths.append(synthesize_style(name, STYLE_PRESETS[name], seed=20260505, num_steps=24))

for path in paths:
    print(path.name)
    display(Audio(filename=str(path)))


In [ ]:
# 追加比較。必要なものだけコメントアウトを外してください。
# extra_paths = []
# for name, caption in STYLE_PRESETS.items():
#     print("\n===", name, "===")
#     extra_paths.append(synthesize_style(name, caption, seed=20260505, num_steps=32))
#
# for path in extra_paths:
#     print(path.name)
#     display(Audio(filename=str(path)))


## 調整のコツ

- 話し方が弱いときは `cfg_scale_caption` を `4.0` から `5.0` くらいに上げる。
- 音が不安定なときは `num_steps` を `32` から `40` に上げる。
- 同じ caption で別の声にしたいときは `seed` を変える。
- まず素早く試すときは `num_steps=12`、本番候補は `num_steps=32` 以上がおすすめ。